# Dataset Exploration
### Dataset: Sleep-EDF Expanded

---

**Objective:** Exploring the dataset, so I'll have a better vision of what kinda data I'm dealing with before the preprocessing stage.

**Steps:**
1. Import the dataset paths
2. Load PSG
3. PSG information
4. Load hypnogram
5. Annotation information
6. Plot EEG + annotations
7. Stage statistics
8. Inspect unknown annotation

## 1. Import the dataset

In [ ]:
from pathlib import Path
import mne

# Change this to your actual Sleep-EDF dataset location
DATA_DIR = Path(r"../../Dataset/sleep-edf-database-expanded-1.0.0/sleep-edf-database-expanded-1.0.0")

psg_path = DATA_DIR / "sleep-cassette" / "SC4001E0-PSG.edf"

raw = mne.io.read_raw_edf(
    psg_path,
    preload=False,
    verbose=True
)

print(raw)

## 2. PSG information

In [ ]:
print("\n--- BASIC INFORMATION ---")
print("Number of channels:", len(raw.ch_names))
print("Sampling frequency:", raw.info["sfreq"], "Hz")
print("Recording duration:", raw.times[-1], "seconds")
print("Recording duration:", raw.times[-1] / 3600, "hours")

print("\n--- CHANNELS ---")
for i, channel in enumerate(raw.ch_names):
    print(i, channel)

## 3. Load hypnogram

In [ ]:
hypnogram_path = DATA_DIR / "sleep-cassette" / "SC4001EC-Hypnogram.edf"

annotations = mne.read_annotations(hypnogram_path)

print(annotations)

## 4. Annotation information

In [ ]:
print("\n--- ANNOTATIONS ---")
print("Number of annotations:", len(annotations))

for onset, duration, description in zip(
    annotations.onset,
    annotations.duration,
    annotations.description
):
    print(
        f"Onset: {onset:8.2f}s | "
        f"Duration: {duration:6.2f}s | "
        f"Stage: {description}"
    )

## 5. Plot EEG + annotations

In [ ]:
import matplotlib.pyplot as plt

# Select the Fpz-Cz EEG channel
eeg = raw.copy().pick(["EEG Fpz-Cz"])

# Plot 2 minutes beginning at the first scored sleep epoch
start_time = 30630
duration = 120

eeg.plot(
    start=start_time,
    duration=duration,
    scalings="auto",
    title="SC4001 - EEG Fpz-Cz around Sleep Onset"
)

plt.show()

In [ ]:
eeg2 = raw.copy().pick(["EEG Pz-Oz"])

eeg2.plot(
    start=30630,
    duration=120,
    scalings="auto",
    title="SC4001 - EEG Pz-Oz around Sleep Onset"
)

plt.show()

In [ ]:
raw.set_annotations(annotations)

raw.plot(
    picks=["EEG Fpz-Cz"],
    start=30600,
    duration=180,
    scalings="auto",
    title="SC4001 - EEG with Sleep-Stage Annotations"
)

plt.show()

## 6. Stage statistics

In [ ]:
from collections import Counter

stage_counts = Counter(annotations.description)

for stage, count in stage_counts.items():
    print(f"{stage}: {count}")